In [12]:
import mlflow
import mlflow.pyfunc
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import shap
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, "../../run")
from const import REPO_PATH
from experiment_config import TRAINGING_CONFIG
sys.path.insert(1, f"{REPO_PATH}")
from src.model.experiment_utils import align_on_keys
import joblib
import os

In [5]:
features_path = f"{TRAINGING_CONFIG['features_path']}/all_combined_features_2017-24.csv"
seasons = sorted(TRAINGING_CONFIG['seasons'])
target_dfs = [pd.read_csv(f"{TRAINGING_CONFIG['processed_data_path']}/{season}/all_target_df.csv") for season in seasons[1:]]
for df in target_dfs:
    df['date'] = pd.to_datetime(df['date'])
target_df = pd.concat(target_dfs, ignore_index=True)
feature_df = pd.read_csv(features_path)
feature_df['date'] = pd.to_datetime(feature_df['date'])
feature_df, target_df = align_on_keys(feature_df, target_df, TRAINGING_CONFIG['key_columns'])

In [9]:
# Set the model name and load the model from MLflow
model_name = "XGBClassifier_home_goals_n_estimators=100_max_depth=5_learning_rate=0.01_gamma=0"  # Change if needed
experiment_name = TRAINGING_CONFIG['experiment_name']
tracking_uri = f"{REPO_PATH}/mlflow"
mlflow.set_tracking_uri(tracking_uri)
client = mlflow.tracking.MlflowClient(tracking_uri)
# Search for the run by exact run name (tag 'mlflow.runName')
runs = client.search_runs(
    experiment_ids=[client.get_experiment_by_name(experiment_name).experiment_id],
    filter_string=f"tags.mlflow.runName = '{model_name}'"
)
if len(runs) == 0:
    raise ValueError(f"No runs found for model name: XBGClassifier_away_goals_n_estimators=200_max_depth=7_learning_rate=0.1_gamma=1")
run_id = runs[0].info.run_id
model_uri = f"runs:/{run_id}/model"
print("Run ID:", run_id)
print("Model URI:", model_uri)
artifacts = client.list_artifacts(run_id)
print("Artifacts:", artifacts)

Run ID: 14ece871e17f4b0d985e8b1cc21ae220
Model URI: runs:/14ece871e17f4b0d985e8b1cc21ae220/model
Artifacts: [<FileInfo: file_size=None, is_dir=True, path='model'>]


In [13]:
def load_joblib_model_from_mlflow(experiment_name, model_name, tracking_uri):
    # Set MLflow tracking URI and get experiment
    mlflow.set_tracking_uri(tracking_uri)
    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        raise ValueError(f"Experiment '{experiment_name}' not found.")

    # Search for the run with the given run name (model_name)
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.mlflow.runName = '{model_name}'",
        output_format="pandas"
    )
    if runs.empty:
        raise ValueError(f"No run found with name '{model_name}' in experiment '{experiment_name}'.")

    run_id = runs.iloc[0]['run_id']

    # Download the 'model' artifact directory
    local_dir = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="model")

    # Find the .joblib file in the directory
    joblib_files = [f for f in os.listdir(local_dir) if f.endswith('.joblib')]
    if not joblib_files:
        raise FileNotFoundError(f"No .joblib file found in model artifact for run {run_id}")
    model_path = os.path.join(local_dir, joblib_files[0])

    # Load the model
    model = joblib.load(model_path)
    print(f"Loaded model from {model_path}")
    return model

In [14]:
model = load_joblib_model_from_mlflow(experiment_name, model_name, tracking_uri)

Loaded model from /var/folders/5j/cgxgswl52bv6qlx0pdt0f8080000gn/T/tmp8cnjkqzw/model/XGBClassifier_home_goals_n_estimators=100_max_depth=5_learning_rate=0.01_gamma=0.joblib


In [18]:
# SHAP value analysis

explainer = shap.Explainer(model.predict_proba, feature_df.drop(columns=['date', 'home', 'away']))
shap_values = explainer(feature_df)
shap.summary_plot(shap_values, feature_df, show=False)
plt.show()

DimensionError: The passed data does not match the background shape expected by the masker! The data of shape (2729,) was passed while the masker expected data of shape (2726,).